# XPLT Explorer — Multi-Case FEBio Post-Processor

Loads one or more `.feb` / `.xplt` simulation cases and computes:
- Facet geometry (centroids, areas)
- Contact pressure time-series per facet
- Contact Surface Area Ratio (CSAR) vs time
- Multi-case CSAR comparison

All heavy parsing and computation lives in **`xplt_core.py`**.  
You only need to edit the **Configuration** cell below.

In [ ]:
import xplt_core as xc
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.dpi'] = 110

## Configuration

Edit this cell to define your simulation cases and analysis parameters.
- Add one `dict` entry per case.  `label` is used in plot titles and file names.
- `contact_surface_name` should be a substring of the xplt surface name (case-insensitive).
- `tip_material_names` determines which material domains define the "tip" Z-range.
  Leave as `None` to fall back to `tip_z_cutoff`.
- `ZMIN` / `ZMAX` set the Z-window for CSAR analysis (millimetres).

In [ ]:
# ── Simulation cases ──────────────────────────────────────────────────────────
# Add as many cases as you like; compare them in the CSAR section below.
CASE_DEFS = [
    dict(
        feb_path             = 'sample.feb',
        xplt_path            = 'sample.xplt',
        label                = 'sample',
        contact_surface_name = 'catheter_slidePrimary',   # substring match, case-insensitive
        tip_material_names   = {'catheter_tip'},           # set of material names → tip Z range
        tip_z_cutoff         = 50.0,                       # fallback if tip materials not found
        cp_var_id            = 1,                          # xplt variable ID for contact pressure
    ),
    # dict(
    #     feb_path             = 'sample2.feb',
    #     xplt_path            = 'sample2.xplt',
    #     label                = 'sample2',
    #     contact_surface_name = 'catheter_slidePrimary',
    #     tip_material_names   = {'catheter_tip'},
    #     tip_z_cutoff         = 50.0,
    #     cp_var_id            = 1,
    # ),
]

# ── CSAR analysis window ───────────────────────────────────────────────────────
# Set to None to use the tip Z range derived from tip_material_names automatically.
ZMIN = None   # mm  (or e.g. 0.0)
ZMAX = None   # mm  (or e.g. 50.0)

## Load Cases

Run this cell once (or re-run after changing `CASE_DEFS`).
Parsing is automatic — no further setup needed.

In [ ]:
cases = [xc.SimulationCase(**d) for d in CASE_DEFS]

## Simulation Summary

In [ ]:
for case in cases:
    print(case.summary())
    print(f'\n  Available xplt surfaces:')
    for sid, name in sorted(case.surface_names.items()):
        print(f'    id={sid:2d}  "{name}"')
    print()

## Facet Geometry

2-D projections of facet centroids. Orange = tip zone, blue = body.
The red dashed lines mark the tip Z boundaries derived from the material domain.

In [ ]:
for case in cases:
    xc.plot_geometry(case)

## Contact Pressure Analysis

4-panel overview per case:
1. Max / mean contact pressure vs time
2. Number of facets in contact vs time  
3. Facet map coloured by peak cp
4. Top-10 tip-zone facets individual cp traces

In [ ]:
for case in cases:
    xc.plot_contact_overview(case)

## Contact Surface Area Ratio (CSAR)

$$\text{CSAR}(t) = \frac{\displaystyle\sum_{i \in \text{region},\; cp_i(t) > 0} A_i}{\displaystyle\sum_{i \in \text{region}} A_i}$$

`ZMIN` / `ZMAX` were set in the Configuration cell.  
If both are `None`, the tip Z range from the material domain is used automatically.

In [ ]:
# CSAR plot for each individual case
for case in cases:
    xc.plot_csar(case, zmin=ZMIN, zmax=ZMAX)

In [ ]:
# Multi-case CSAR overlay — meaningful only when more than one case is loaded
xc.compare_csar(cases, zmin=ZMIN, zmax=ZMAX)

# Numeric CSAR table (columns = case labels, rows = timesteps)
df_csar = xc.csar_table(cases, zmin=ZMIN, zmax=ZMAX)
print(df_csar.to_string(index=False))

## Multi-Band CSAR

Split the Z-axis into discrete bands and compute CSAR for each band individually,
plus an **accumulated total** across all bands (union — no double-counting).

- Edit `Z_BANDS` to define your regions of interest.
- `BAND_LABELS` are optional display names; set to `None` to auto-generate.
- `TOTAL_AREA_OVERRIDE` lets you normalise the accumulated CSAR against a known
  reference area (e.g. from CAD) instead of the sum of facet areas in the bands.

In [ ]:
# ── Band definitions ───────────────────────────────────────────────────────────
# Each tuple is (zmin, zmax) in mm.  Adjust to match your anatomy / mesh.
Z_BANDS = [
    ( 0.0, 10.0),   # proximal
    (10.0, 20.0),   # mid
    (20.0, 30.0),   # distal
]

# Optional human-readable names for each band (must match len(Z_BANDS) or be None).
BAND_LABELS = ['Proximal (0–10 mm)', 'Mid (10–20 mm)', 'Distal (20–30 mm)']

# Set to a float [mm²] to normalise the accumulated CSAR against a fixed reference
# area (e.g. from CAD).  None → use the sum of facet areas inside the union of bands.
TOTAL_AREA_OVERRIDE = None

# ── Per-case multi-band plot ───────────────────────────────────────────────────
for case in cases:
    # Raw band statistics — inspect before plotting if needed
    ts, band_stats, accumulated = case.compute_region_accumulation(Z_BANDS)

    print(f'\n{case.label} — band summary:')
    for bs, lbl in zip(band_stats, BAND_LABELS or [f'band {i}' for i in range(len(band_stats))]):
        print(f'  {lbl:30s}  {bs["n_facets_in_region"]:5d} facets  '
              f'A = {bs["total_area_mm2"]:.2f} mm²')
    denom = TOTAL_AREA_OVERRIDE or accumulated['total_area_mm2']
    print(f'  {"Accumulated (union)":30s}  {accumulated["n_facets_in_region"]:5d} facets  '
          f'A = {denom:.2f} mm²')

    xc.plot_csar_multi_regions(case, Z_BANDS, band_labels=BAND_LABELS,
                               total_area_override=TOTAL_AREA_OVERRIDE)

# ── Multi-case accumulated CSAR comparison ─────────────────────────────────────
# Meaningful only when more than one case is loaded.
# Provide one override per case, or None to use computed area.
xc.compare_csar_accumulated(cases, Z_BANDS, band_labels=BAND_LABELS,
                             total_area_overrides=[TOTAL_AREA_OVERRIDE] * len(cases))

# ── Numeric table: accumulated CSAR across all bands ──────────────────────────
df_csar_multi = xc.csar_accumulated_table(cases, Z_BANDS)
print('\nAccumulated CSAR table (first 5 rows):')
print(df_csar_multi.head().to_string(index=False))

## Export

Uncomment whichever exports you need.

In [ ]:
# ── VTP + PVD series (ParaView animation) ─────────────────────────────────────
# Each case exports to its own sub-directory: {label}_vtp/
# for case in cases:
#     case.export_vtp()

# ── Per-facet summary CSV ──────────────────────────────────────────────────────
for case in cases:
    csv_path = f'{case.label}_facet_summary.csv'
    case.df_facets.to_csv(csv_path, index=False)
    print(f'Saved: {csv_path}')

# ── CSAR table CSV ─────────────────────────────────────────────────────────────
df_csar.to_csv('csar_comparison.csv', index=False)
print('Saved: csar_comparison.csv')

In [ ]:
# ── Surrogate-lab compatible CSV export ───────────────────────────────────────
# Exports one row per (facet × timestep) with column names matching surrogate-lab.
# insertion_depth is computed automatically from prescribed z-displacement BCs
# in the .feb file — no manual depth_scale needed.
#
# To add a new variable later:
#   1. Add a _col_<name>(self) method in SimulationCase (xplt_core.py)
#   2. Register it in SimulationCase.SURROGATE_COLUMNS
#   3. Add the column name to surrogate-lab's configs/config.yaml features.inputs

for case in cases:
    df_sl = case.df_surrogate()
    out_path = f'{case.label}_surrogate.csv'
    df_sl.to_csv(out_path, index=False)
    print(f'Saved: {out_path}  ({len(df_sl):,} rows)')
    print(f'  Columns : {list(df_sl.columns)}')
    print(f'  Depth range: {df_sl["insertion_depth"].min():.2f} … {df_sl["insertion_depth"].max():.2f} mm')
